# 저번주 코드에서 개선 시킨 점
1. 각 시점에서 출력값을 내는건 forward 메소드안에서 하지 않았지만 출력값을 내는것까지 메소드안에서 처리하였습니다
2. 각 기울기를 합산하여 가중치를 업데이트하기 위해 기울기를 합산할 grads_sum 속성 추가 (교재 코드에는 없던 부분)
3. cache에는 역전파시에 필요한 세 가지 요소가 저장되는데 cache가 하나만 있으면 마지막 시점의 것만 저장되는 문제를 발견하여 caches 리스트를 따로 만들었습니다

In [1]:
import numpy as np

class RNN: 
    def __init__(self, Wx,Wh,b_h,Why,b_y):
        self.params = [Wx,Wh,b_h,Why,b_y]
        self.grads = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b_h),np.zeros_like(Why),np.zeros_like(b_y)]
        self.grads_sum = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b_h),np.zeros_like(Why),np.zeros_like(b_y)]
        self.cache = None

    def forward(self, x, h_prev):
        Wx, Wh, b_h, Why, b_y = self.params
        t = np.dot(Wh,h_prev) + np.dot(Wx,x) + b_h
        h_next = np.tanh(t)
        y = h_next.T @ Why + b_y
        self.cache = (x,h_prev,h_next)
        return h_next, y

    def error(self,y,target):

        return np.sum((target-y)**2).round(4)

    def backward(self, y, dh_next):
        Wx, Wh, b_h, Why, b_y = self.params
        x,h_prev,h_next = self.cache

        dy = 2*y
        dh = dy*Why + dh_next
        dt = dh*(1-h_next**2)
        db_h = dt
        dWh = np.dot(dt,h_prev.T)
        dh_prev = np.dot(Wh.T,dt)
        dWx = np.dot(dt,x.T)
        dWhy = np.dot(dy,h_next.T)
        db_y = dy

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db_h
        self.grads[3][...] = dWhy.T
        self.grads[4][...] = db_y

        self.grads_sum[0][...] += dWx
        self.grads_sum[1][...] += dWh
        self.grads_sum[2][...] += db_h
        self.grads_sum[3][...] += dWhy.T
        self.grads_sum[4][...] += db_y.reshape(-1)

        return dh_prev,self.grads
    
    def update(self,lr):

        for i, grad in enumerate(self.grads_sum):
            self.params[i] -= lr*self.grads_sum[i]
        return self.params





Wx = np.array([[0.1, 0.2],
               [0.3, 0.4]])   
Wh = np.array([[0.5, 0.0],
               [0.0, 0.5]])   
b_h = np.array([[0.1, 0.1]]).T
  
 
# 출력층
Why = np.array([0.5, 0.5]).reshape(-1,1)   
b_y = np.array([0.1])
 

 
xs = [
    np.array([[1.0, 2.0]]).T,  # x_1 
    np.array([[0.0, 1.0]]).T,  # x_2
    np.array([[1.0, 1.0]]).T,  # x_3
]
 

rnn = RNN(Wx, Wh, b_h,Why,b_y)
target = 0 #목표값
rep = 20 #학습횟수
lr = 0.05 #학습률
errors = []
for _ in range(rep - 1):
    caches = []
    grads = []
    h = np.zeros((2, 1))  # h_0
    ys = []
    print('='*30)
    print(f"순전파 {_ + 1} 회차")
    for t, x in enumerate(xs, start=1):
        h,y_t = rnn.forward(x, h)
        ys.append(y_t)
        caches.append(rnn.cache)
        print(f"t_{t} h={str(h[0].round(4).T)} y={ys[t - 1].round(4).T}\n")
   
    e = rnn.error(np.array(ys),target)
    errors.append(e)
    print(f'error : {e}')

    dh_next = np.zeros((2,1))
    rnn.grads_sum = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b_h),np.zeros_like(Why),np.zeros_like(b_y)]
    print('='*30)
    print(f"역전파 {_ + 1} 회차")
    for t in reversed(range(3)):
        rnn.cache = caches[t]
        dh_next, grad = rnn.backward(ys[t],dh_next)
        print(f"t_{t + 1}")
        print(f'dWx:{grad[0].round(4)}')
        print(f'dWh:{grad[1].round(4)}')
        print(f'db_h:{grad[2].round(4)}')
        print(f'dWhy:{grad[3].round(4)}')
        print(f'db_y:{grad[4].round(4)}\n')

    
    rnn.update(lr)

h = np.zeros((2, 1))
ys = []
print('='*30)
print(f"순전파 {rep} 회차")
for t, x in enumerate(xs, start=1):
    h,y_t = rnn.forward(x, h)
    ys.append(y_t)
    print(f"t_{t} h={str(h[0].round(4).T)} y={ys[t - 1].round(4).T}\n")
   

e = rnn.error(np.array(ys),target)
errors.append(e)
print(f'error : {e}')

print('='*30)
print("오차 변화 요약")
for i, error in enumerate(errors):
    print(f'{i + 1}회차 오차{error}')


순전파 1 회차
t_1 h=[0.537] y=[[0.7854]]

t_2 h=[0.5143] y=[[0.7193]]

t_3 h=[0.5765] y=[[0.7991]]

error : 1.7728
역전파 1 회차
t_3
dWx:[[0.5336 0.5336]
 [0.2595 0.2595]]
dWh:[[0.2744 0.3865]
 [0.1334 0.188 ]]
db_h:[[0.5336]
 [0.2595]]
dWhy:[[0.9213]
 [1.3133]]
db_y:[1.5982]

t_2
dWx:[[0.     0.7253]
 [0.     0.4035]]
dWh:[[0.3895 0.6047]
 [0.2167 0.3364]]
db_h:[[0.7253]
 [0.4035]]
dWhy:[[0.7399]
 [1.0422]]
db_y:[1.4387]

t_1
dWx:[[0.8169 1.6338]
 [0.3011 0.6022]]
dWh:[[0. 0.]
 [0. 0.]]
db_h:[[0.8169]
 [0.3011]]
dWhy:[[0.8435]
 [1.3094]]
db_y:[1.5707]

순전파 2 회차
t_1 h=[0.1385] y=[[0.1624]]

t_2 h=[0.0784] y=[[0.0995]]

t_3 h=[0.089] y=[[0.1386]]

error : 0.0555
역전파 2 회차
t_3
dWx:[[0.1031 0.1031]
 [0.0392 0.0392]]
dWh:[[0.0081 0.0652]
 [0.0031 0.0248]]
db_h:[[0.1031]
 [0.0392]]
dWhy:[[0.0247]
 [0.2062]]
db_y:[0.2772]

t_2
dWx:[[0.     0.1213]
 [0.     0.0459]]
dWh:[[0.0168 0.0922]
 [0.0064 0.0349]]
db_h:[[0.1213]
 [0.0459]]
dWhy:[[0.0156]
 [0.126 ]]
db_y:[0.199]

t_1
dWx:[[0.1741 0.3482]
 [0.05   

# 기울기 소실 (vanishing gradient) 문제를 보이기 위한 예제
- 마지막 시점에만 오차가 있는거 처럼 보이기 위해 마지막 시점을 제외하고 나머지는 다 y를 0으로 만듭니다
- 이러면 역전파를 할 때 마지막 시점을 제외한 나머지 시점은 y가 0으로 들어가 dy도 0이 됩니다
- dh는 각 시점이 마지막 시점의 은닉상태가 손실에 얼마나 기여하는지 알 수 있습니다
- dh는 이후에 더 계산을 거쳐 dh_prev가 이전 시점으로 전달되는데 이 값은 tanh 미분값과 Wh가 반복적으로 곱해서 결국 0이되는것을 알 수 있습니다
- backward 메소드가 dh를 반환하도록 수정하였습니다.

In [2]:
import numpy as np

class RNN: 
    def __init__(self, Wx,Wh,b_h,Why,b_y):
        self.params = [Wx,Wh,b_h,Why,b_y]
        self.grads = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b_h),np.zeros_like(Why),np.zeros_like(b_y)]
        self.grads_sum = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b_h),np.zeros_like(Why),np.zeros_like(b_y)]
        self.cache = None

    def forward(self, x, h_prev):
        Wx, Wh, b_h, Why, b_y = self.params
        t = np.dot(Wh,h_prev) + np.dot(Wx,x) + b_h
        h_next = np.tanh(t)
        y = h_next.T @ Why + b_y
        self.cache = (x,h_prev,h_next)
        return h_next, y

    def error(self,y,target):

        return np.sum((target-y)**2).round(4)

    def backward(self, y, dh_next):
        Wx, Wh, b_h, Why, b_y = self.params
        x,h_prev,h_next = self.cache

        dy = 2*y
        dh = dy*Why + dh_next
        dt = dh*(1-h_next**2)
        db_h = dt
        dWh = np.dot(dt,h_prev.T)
        dh_prev = np.dot(Wh.T,dt)
        dWx = np.dot(dt,x.T)
        dWhy = np.dot(dy,h_next.T)
        db_y = dy

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db_h
        self.grads[3][...] = dWhy.T
        self.grads[4][...] = db_y

        self.grads_sum[0][...] += dWx
        self.grads_sum[1][...] += dWh
        self.grads_sum[2][...] += db_h
        self.grads_sum[3][...] += dWhy.T
        self.grads_sum[4][...] += db_y.reshape(-1)

        return dh,dh_prev,self.grads
    def update(self,lr):

        for i, grad in enumerate(self.grads_sum):
            self.params[i] -= lr*self.grads_sum[i]
        return self.params





Wx = np.array([[0.1, 0.2],
               [0.3, 0.4]])   
Wh = np.array([[0.5, 0.0],
               [0.0, 0.5]])   
b_h = np.array([[0.1, 0.1]]).T
  
 
# 출력층
Why = np.array([0.5, 0.5]).reshape(-1,1)   
b_y = np.array([0.1])
 

 
base = [np.array([[1.0,2.0]]).T, np.array([[0.0,1.0]]).T, np.array([[1.0,1.0]]).T]
T = 21
xs = [base[i % 3] for i in range(T)]
 

rnn = RNN(Wx, Wh, b_h,Why,b_y)
target = 0 #목표값
rep = 3 #학습횟수
lr = 0.05 #학습률
errors = []
# ---- 순전파 (T스텝 전부) ----
caches = []
h = np.zeros((2, 1))
ys = []
for x in xs:
    h, y_t = rnn.forward(x, h)
    ys.append(y_t)
    caches.append(rnn.cache)
 
print(f"마지막 시점(t={T}) y = {ys[-1].round(4)}")
 
# ---- 역전파: 손실은 "맨 마지막 시점"에만 있는 것처럼 처리 ----
dh_next = np.zeros((2, 1))
for t in reversed(range(T)):
    rnn.cache = caches[t]
    y_input = ys[t] if t == T - 1 else np.zeros_like(ys[t])
    dh, dh_next, grad = rnn.backward(y_input, dh_next)

    print(f"t_{t+1}의 dh\n{dh.round(4)}")

마지막 시점(t=21) y = [[0.811]]
t_21의 dh
[[0.811]
 [0.811]]
t_20의 dh
[[0.261 ]
 [0.1295]]
t_19의 dh
[[0.0871]
 [0.0288]]
t_18의 dh
[[0.0212]
 [0.0021]]
t_17의 dh
[[0.0068]
 [0.0003]]
t_16의 dh
[[0.0023]
 [0.0001]]
t_15의 dh
[[0.0006]
 [0.    ]]
t_14의 dh
[[0.0002]
 [0.    ]]
t_13의 dh
[[0.0001]
 [0.    ]]
t_12의 dh
[[0.]
 [0.]]
t_11의 dh
[[0.]
 [0.]]
t_10의 dh
[[0.]
 [0.]]
t_9의 dh
[[0.]
 [0.]]
t_8의 dh
[[0.]
 [0.]]
t_7의 dh
[[0.]
 [0.]]
t_6의 dh
[[0.]
 [0.]]
t_5의 dh
[[0.]
 [0.]]
t_4의 dh
[[0.]
 [0.]]
t_3의 dh
[[0.]
 [0.]]
t_2의 dh
[[0.]
 [0.]]
t_1의 dh
[[0.]
 [0.]]
